# 03 - Spillover Index de Diebold-Yilmaz

Neste notebook, exploramos o framework de **spillover** e **connectedness**
desenvolvido por Diebold e Yilmaz para medir interdependencias dinamicas
entre series temporais.

### Conteudo
- Spillover index: full-sample e rolling window
- Connectedness table (FROM, TO, NET)
- Generalized FEVD (Koop-Pesaran-Potter)
- Visualizacoes: heatmap, network, rolling
- Exercicios praticos

### Referencias
- Diebold, F.X. & Yilmaz, K. (2009). *Measuring Financial Asset Return and Volatility Spillovers, with Application to Global Equity Markets*. Economic Journal, 119(534), 158-171.
- Diebold, F.X. & Yilmaz, K. (2012). *Better to Give than to Receive: Predictive Directional Measurement of Volatility Spillovers*. International Journal of Forecasting, 28(1), 57-66.
- Koop, G., Pesaran, M.H. & Potter, S.M. (1996). *Impulse Response Analysis in Nonlinear Multivariate Models*. Journal of Econometrics, 74(1), 119-147.
- Pesaran, H.H. & Shin, Y. (1998). *Generalized Impulse Response Analysis in Linear Multivariate Models*. Economics Letters, 58(1), 17-29.

## 1. Setup e Carregamento dos Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

# Adicionar o diretorio raiz do projeto ao path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Imports do chronobox
from chronobox.analysis.spillover import SpilloverIndex
from chronobox.visualization.spillover_plot import plot_heatmap, plot_network, plot_rolling

# Helpers
sys.path.insert(0, os.path.join(project_root, 'examples', 'filters'))
from utils.plot_helpers import plot_spillover_heatmap
from utils.data_generators import generate_multivariate_cycle

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

print('Setup completo!')

## 2. Teoria: Spillover Index

### Intuicao

O spillover index mede **quanto da variancia de previsao** de uma variavel
pode ser atribuida a choques em **outras** variaveis do sistema.

### Framework

1. Estimar um modelo VAR($p$): $\mathbf{y}_t = \sum_{i=1}^{p} \mathbf{A}_i \mathbf{y}_{t-i} + \mathbf{u}_t$

2. Computar a representacao VMA($\infty$): $\mathbf{y}_t = \sum_{i=0}^{\infty} \mathbf{\Phi}_i \mathbf{u}_{t-i}$

3. Calcular a **Generalized FEVD** (Koop-Pesaran-Potter, 1996; Pesaran-Shin, 1998):

$$\tilde{\theta}_{ij}(H) = \frac{\sigma_{jj}^{-1} \sum_{h=0}^{H-1} (\mathbf{e}_i' \mathbf{\Phi}_h \mathbf{\Sigma} \mathbf{e}_j)^2}{\sum_{h=0}^{H-1} (\mathbf{e}_i' \mathbf{\Phi}_h \mathbf{\Sigma} \mathbf{\Phi}_h' \mathbf{e}_i)}$$

4. Normalizar para que as linhas somem 100%:

$$d_{ij}(H) = \frac{\tilde{\theta}_{ij}(H)}{\sum_{j=1}^{K} \tilde{\theta}_{ij}(H)} \times 100$$

### Medidas de Connectedness

- **Total Spillover**: $S(H) = \frac{\sum_{i \neq j} d_{ij}}{K} \times 100$
- **FROM**: quanto a variavel $i$ **recebe** das outras: $S_{i\leftarrow \bullet} = \sum_{j \neq i} d_{ij}$
- **TO**: quanto a variavel $i$ **transmite** para as outras: $S_{\bullet \leftarrow i} = \sum_{j \neq i} d_{ji}$
- **NET**: $S_i^{\text{net}} = S_{\bullet \leftarrow i} - S_{i \leftarrow \bullet}$ (positivo = transmissor liquido)

## 3. Dados Sinteticos com Fator Comum

In [ ]:
# Gerar 4 series com fator ciclico comum (peso 0.6)
df_multi = generate_multivariate_cycle(n=300, k=4, common_factor_weight=0.6, seed=42)

var_names = ['Var 1', 'Var 2', 'Var 3', 'Var 4']
data_cols = ['var_1', 'var_2', 'var_3', 'var_4']

print(f'Dimensoes: {df_multi[data_cols].shape}')
print(f'Periodo: {df_multi["date"].iloc[0].date()} a {df_multi["date"].iloc[-1].date()}')

# Visualizar as series
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for i, (col, name) in enumerate(zip(data_cols, var_names)):
    ax = axes.ravel()[i]
    ax.plot(df_multi['date'], df_multi[col], linewidth=0.8)
    ax.set_title(name)
    ax.grid(True, alpha=0.3)

fig.suptitle('Series Sinteticas com Fator Ciclico Comum', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Spillover Index: Analise Full-Sample

In [ ]:
# Preparar dados (matrix T x K)
data = df_multi[data_cols].values

# Criar o spillover index com VAR(2) e horizonte de 10 periodos
sp = SpilloverIndex(lags=2, horizon=10)
result = sp.fit(data)

# Resumo textual
print(result.summary())

In [ ]:
# Grafico 1: Connectedness table como heatmap
fig = plot_heatmap(
    spillover=result,
    var_names=var_names,
    title='Tabela de Connectedness (Spillover)'
)
plt.show()

In [ ]:
# Connectedness table detalhada
print('=== Connectedness Table ===')
print(f'{"":>10}', end='')
for name in var_names:
    print(f'{name:>10}', end='')
print(f'{"FROM":>10}')
print('-' * (10 + 10 * (len(var_names) + 1)))

for i, name in enumerate(var_names):
    print(f'{name:>10}', end='')
    for j in range(len(var_names)):
        print(f'{result.fevd_table[i, j]:>10.2f}', end='')
    print(f'{result.directional_from[i]:>10.2f}')

print('-' * (10 + 10 * (len(var_names) + 1)))
print(f'{"TO":>10}', end='')
for j in range(len(var_names)):
    print(f'{result.directional_to[j]:>10.2f}', end='')
print(f'{result.total_spillover:>10.2f}')

print(f'\n{"NET":>10}', end='')
for j in range(len(var_names)):
    print(f'{result.net_spillover[j]:>10.2f}', end='')
print()

In [ ]:
# Grafico 2: Directional spillovers (FROM, TO, NET)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

x = np.arange(len(var_names))
width = 0.5

# FROM others
axes[0].bar(x, result.directional_from, width, color='tab:blue', alpha=0.7)
axes[0].set_xticks(x)
axes[0].set_xticklabels(var_names)
axes[0].set_title('FROM Others (receptividade)')
axes[0].set_ylabel('Spillover (%)')
axes[0].grid(True, alpha=0.3, axis='y')

# TO others
axes[1].bar(x, result.directional_to, width, color='tab:orange', alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels(var_names)
axes[1].set_title('TO Others (transmissao)')
axes[1].set_ylabel('Spillover (%)')
axes[1].grid(True, alpha=0.3, axis='y')

# NET
colors = ['tab:green' if n > 0 else 'tab:red' for n in result.net_spillover]
axes[2].bar(x, result.net_spillover, width, color=colors, alpha=0.7)
axes[2].set_xticks(x)
axes[2].set_xticklabels(var_names)
axes[2].set_title('NET Spillover (TO - FROM)')
axes[2].set_ylabel('Spillover (%)')
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].grid(True, alpha=0.3, axis='y')

fig.suptitle(f'Spillover Direcional (Total = {result.total_spillover:.1f}%)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Grafico 3: Network plot (spillover entre pares)
fig = plot_network(
    spillover=result,
    var_names=var_names,
    threshold=1.0,
    title='Rede de Spillover'
)
plt.show()

## 5. Rolling Window Spillover

O spillover pode variar ao longo do tempo. A analise rolling window computa
o indice total para cada janela de $W$ observacoes, revelando periodos de
maior ou menor interconectividade.

In [ ]:
# Rolling spillover com janela de 100 observacoes
rolling_result = sp.rolling(data, window=100)

print(f'Rolling spillover calculado:')
print(f'  Janela: {rolling_result.window_size} observacoes')
print(f'  Numero de janelas: {len(rolling_result.total_spillover)}')
print(f'  Total spillover - media: {rolling_result.total_spillover.mean():.2f}%')
print(f'  Total spillover - min: {rolling_result.total_spillover.min():.2f}%')
print(f'  Total spillover - max: {rolling_result.total_spillover.max():.2f}%')

In [ ]:
# Grafico 4: Rolling total spillover
fig = plot_rolling(
    rolling_spillover=rolling_result,
    title='Total Spillover Index (Rolling Window = 100)'
)
plt.show()

In [ ]:
# Grafico 5: Rolling directional FROM para cada variavel
n_windows = rolling_result.directional_from.shape[0]
window_dates = df_multi['date'].iloc[-n_windows:].values

fig, ax = plt.subplots(figsize=(14, 5))
for i, name in enumerate(var_names):
    ax.plot(window_dates, rolling_result.directional_from[:, i], label=name, linewidth=1)

ax.set_title('Rolling Directional FROM Others (por variavel)')
ax.set_ylabel('Spillover FROM (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Grafico 6: Rolling NET spillover
fig, ax = plt.subplots(figsize=(14, 5))
for i, name in enumerate(var_names):
    ax.plot(window_dates, rolling_result.net_spillover[:, i], label=name, linewidth=1)

ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_title('Rolling NET Spillover (positivo = transmissor liquido)')
ax.set_ylabel('NET Spillover (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Sensibilidade aos Parametros

### 6.1 Efeito do horizonte de previsao

In [ ]:
# Grafico 7: Total spillover por horizonte
horizons = [2, 5, 10, 20, 50]
total_by_horizon = []

for h in horizons:
    sp_h = SpilloverIndex(lags=2, horizon=h)
    res_h = sp_h.fit(data)
    total_by_horizon.append(res_h.total_spillover)
    print(f'  Horizonte H={h:>2}: Total Spillover = {res_h.total_spillover:.2f}%')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(horizons, total_by_horizon, 'o-', linewidth=2, markersize=8)
ax.set_xlabel('Horizonte de Previsao (H)')
ax.set_ylabel('Total Spillover (%)')
ax.set_title('Sensibilidade do Total Spillover ao Horizonte')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.2 Efeito do numero de lags do VAR

In [ ]:
# Total spillover por numero de lags
lags_list = [1, 2, 3, 4, 6, 8]
total_by_lags = []

for lag in lags_list:
    sp_l = SpilloverIndex(lags=lag, horizon=10)
    res_l = sp_l.fit(data)
    total_by_lags.append(res_l.total_spillover)
    print(f'  VAR({lag}): Total Spillover = {res_l.total_spillover:.2f}%')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(lags_list, total_by_lags, 's-', linewidth=2, markersize=8, color='tab:orange')
ax.set_xlabel('Numero de Lags (p)')
ax.set_ylabel('Total Spillover (%)')
ax.set_title('Sensibilidade do Total Spillover ao Numero de Lags')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Pairwise Spillover

O spillover pairwise mede a transmissao liquida entre cada par de variaveis:

$$C_{ij} = d_{ji} - d_{ij}$$

Se $C_{ij} > 0$, a variavel $i$ e transmissor liquido para $j$.

In [ ]:
# Pairwise net spillover
fig = plot_spillover_heatmap(
    result.pairwise_spillover,
    var_names,
    title='Pairwise Net Spillover (positivo = i transmite para j)'
)
plt.show()

print('Pairwise Net Spillover:')
for i in range(len(var_names)):
    for j in range(len(var_names)):
        if i != j:
            val = result.pairwise_spillover[i, j]
            direction = 'transmite para' if val > 0 else 'recebe de'
            print(f'  {var_names[i]} -> {var_names[j]}: {val:+.2f} ({var_names[i]} {direction} {var_names[j]})')

## Exercicio 1: Spillover entre PIB dos EUA e Brasil

Combine os dados de PIB dos EUA e do Brasil e analise o spillover entre eles:

1. Carregue `us_gdp_quarterly.csv` e `brazil_gdp.csv`
2. Alinhe os periodos em comum
3. Compute o spillover index (VAR(2), horizonte=10)
4. Quem e o transmissor liquido? Faz sentido economico?

In [ ]:
# Exercicio 1: Spillover EUA-Brasil
data_dir = os.path.join(project_root, 'examples', 'filters', 'data')

us_gdp = pd.read_csv(os.path.join(data_dir, 'us_gdp_quarterly.csv'), parse_dates=['date'])
br_gdp = pd.read_csv(os.path.join(data_dir, 'brazil_gdp.csv'), parse_dates=['date'])

# Alinhar periodos
merged = pd.merge(us_gdp[['date', 'gdp_log']], br_gdp[['date', 'gdp_log']],
                   on='date', suffixes=('_us', '_br'))

# Usar primeiras diferencas (retornos) para estacionariedade
gdp_returns = merged[['gdp_log_us', 'gdp_log_br']].diff().dropna().values

print(f'Periodo em comum: {merged["date"].iloc[0].date()} a {merged["date"].iloc[-1].date()}')
print(f'Observacoes: {len(gdp_returns)}')

# Spillover
sp_gdp = SpilloverIndex(lags=2, horizon=10)
result_gdp = sp_gdp.fit(gdp_returns)

gdp_names = ['PIB EUA', 'PIB Brasil']
print(f'\nTotal Spillover: {result_gdp.total_spillover:.2f}%')
print(f'\nNET spillover:')
for i, name in enumerate(gdp_names):
    net = result_gdp.net_spillover[i]
    role = 'transmissor liquido' if net > 0 else 'receptor liquido'
    print(f'  {name}: {net:+.2f} ({role})')

# Heatmap
fig = plot_heatmap(
    spillover=result_gdp,
    var_names=gdp_names,
    title='Spillover: PIB EUA vs PIB Brasil'
)
plt.show()

## Exercicio 2: Efeito do Tamanho da Janela no Rolling Spillover

1. Compute o rolling spillover para os dados sinteticos com janelas de 50, 100 e 150
2. Compare os graficos do total spillover ao longo do tempo
3. Qual janela produz uma serie mais suave? Qual capta melhor mudancas abruptas?

In [ ]:
# Exercicio 2: Rolling spillover com diferentes janelas
fig, ax = plt.subplots(figsize=(14, 5))

for window in [50, 100, 150]:
    roll = sp.rolling(data, window=window)
    n_w = len(roll.total_spillover)
    w_dates = df_multi['date'].iloc[-n_w:].values
    ax.plot(w_dates, roll.total_spillover, label=f'W={window}', linewidth=1.2)

ax.set_title('Rolling Total Spillover: Efeito do Tamanho da Janela')
ax.set_ylabel('Total Spillover (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercicio 3: Spillover com Maior Interdependencia

1. Gere dados com `common_factor_weight=0.9` (alta interdependencia)
2. Gere dados com `common_factor_weight=0.2` (baixa interdependencia)
3. Compare o total spillover dos dois cenarios
4. Como o fator comum afeta a connectedness?

In [ ]:
# Exercicio 3: Alta vs baixa interdependencia
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, weight, label in zip(axes, [0.9, 0.2], ['Alta (w=0.9)', 'Baixa (w=0.2)']):
    df_w = generate_multivariate_cycle(n=300, k=4, common_factor_weight=weight, seed=42)
    data_w = df_w[data_cols].values
    
    sp_w = SpilloverIndex(lags=2, horizon=10)
    res_w = sp_w.fit(data_w)
    
    im = ax.imshow(res_w.fevd_table, cmap='YlOrRd', aspect='auto', vmin=0, vmax=100)
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(var_names, rotation=45, ha='right')
    ax.set_yticklabels(var_names)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{res_w.fevd_table[i,j]:.1f}', ha='center', va='center', fontsize=9)
    ax.set_title(f'{label}\nTotal = {res_w.total_spillover:.1f}%')

plt.tight_layout()
plt.show()

## 8. Conclusoes

- O **Spillover Index de Diebold-Yilmaz** quantifica interdependencias entre series via GFEVD
- A abordagem **generalizada** (Koop-Pesaran-Potter) e invariante a ordenacao das variaveis
- A **connectedness table** resume FROM, TO e NET spillover para cada variavel
- O **rolling window** revela como a interconectividade evolui ao longo do tempo
- Parametros-chave: numero de lags (VAR), horizonte de previsao, tamanho da janela

### Aplicacoes tipicas
- Contágio financeiro entre mercados
- Transmissao de choques macroeconomicos entre paises
- Interdependencia entre setores da economia
- Analise de risco sistemico